# 18r — June 2026 market–weather exact common support

This notebook constructs the exact overlap between:

- the verified 18p Polymarket no-look-ahead decision panel; and
- the verified 18q issue-time-admissible deterministic ECMWF forecast panel.

The join key is exactly:

`event_date × market_id × decision_rule`.

A contract-rule row enters common support only when:

1. a Polymarket YES price exists at or before the decision cut-off; and
2. the selected ECMWF run provides all 24 non-missing Hong Kong local event-day hours.

The notebook:

- audits all 1,320 June contract-rule candidates;
- keeps market and weather missingness separate;
- creates the exact common-support contract panel;
- verifies that retained rows form complete eleven-contract books;
- normalises market probabilities only within complete retained books;
- recomputes market binary, categorical and multiclass scores on the exact support;
- summarises deterministic weather errors on the same date-rule support;
- compares the market modal contract and deterministic point-forecast contract descriptively;
- does not convert the deterministic forecast into a probability;
- applies no Gaussian bridge and creates no artificial ensemble features.

The market remains an external probabilistic benchmark. The deterministic forecast maximum remains an input for later local residual post-processing.

The notebook never creates a branch, commit, push, pull request, reminder or notification.

In [1]:
from __future__ import annotations

import hashlib
import json
import math
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / ".git").exists():
    raise RuntimeError(
        "Run this notebook from the repository root. "
        f"Current directory: {REPO_ROOT}"
    )

STEP = "18r"
UTC = timezone.utc
LOG_EPSILON = 1e-6

DECISION_RULE_ORDER = [
    "24h_prior",
    "12h_prior",
    "6h_prior",
    "event_day_open",
]
RULE_ORDER_MAP = {
    rule: index
    for index, rule in enumerate(DECISION_RULE_ORDER)
}

EXPECTED_COMMON_ROWS = {
    "24h_prior": 297,
    "12h_prior": 308,
    "6h_prior": 319,
    "event_day_open": 330,
}
EXPECTED_COMMON_BOOKS = {
    "24h_prior": 27,
    "12h_prior": 28,
    "6h_prior": 29,
    "event_day_open": 30,
}

MARKET_DECISION_PATH = (
    REPO_ROOT
    / "data/processed/18p_june_2026_clob_market_price_recovery"
    / "18p_june_2026_no_lookahead_decision_panel.csv"
)
MARKET_SCORING_PATH = (
    REPO_ROOT
    / "data/processed/18p_june_2026_clob_market_price_recovery"
    / "18p_june_2026_market_scoring_panel.csv"
)
MARKET_SUMMARY_PATH = (
    REPO_ROOT
    / "data/processed/18p_june_2026_clob_market_price_recovery"
    / "18p_june_2026_market_recovery_summary.json"
)

WEATHER_MEMBERSHIP_PATH = (
    REPO_ROOT
    / "data/processed/18q_june_2026_ecmwf_single_run_forecasts"
    / "18q_june_2026_contract_event_membership.csv"
)
WEATHER_DAILY_PATH = (
    REPO_ROOT
    / "data/processed/18q_june_2026_ecmwf_single_run_forecasts"
    / "18q_june_2026_daily_max_forecasts.csv"
)
WEATHER_SUMMARY_PATH = (
    REPO_ROOT
    / "data/processed/18q_june_2026_ecmwf_single_run_forecasts"
    / "18q_june_2026_weather_ingestion_summary.json"
)

OUTCOME_PATH = (
    REPO_ROOT
    / "data/processed/18o_june_2026_hko_realised_outcomes"
    / "18o_june_2026_contract_outcomes.csv"
)

OUT_DIR = (
    REPO_ROOT
    / "data/processed/18r_june_2026_market_weather_common_support"
)
REPORT_DIR = (
    REPO_ROOT
    / "reports/18r_june_2026_market_weather_common_support"
)

for directory in (OUT_DIR, REPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

required_inputs = [
    MARKET_DECISION_PATH,
    MARKET_SCORING_PATH,
    MARKET_SUMMARY_PATH,
    WEATHER_MEMBERSHIP_PATH,
    WEATHER_DAILY_PATH,
    WEATHER_SUMMARY_PATH,
    OUTCOME_PATH,
]

for required_input in required_inputs:
    if not required_input.is_file():
        raise FileNotFoundError(
            f"Required verified input is missing: {required_input}"
        )

print(f"Repository root: {REPO_ROOT}")
print("Join key: event_date × market_id × decision_rule")
print("Probability bridge applied: False")

Repository root: /Users/edwardlee/Desktop/2026MScWeatherForecastingPolymarket
Join key: event_date × market_id × decision_rule
Probability bridge applied: False


In [2]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def parse_boolean(series: pd.Series, *, name: str) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    parsed = (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map(
            {
                "true": True,
                "false": False,
                "1": True,
                "0": False,
            }
        )
    )

    if parsed.isna().any():
        bad = series.loc[parsed.isna()].drop_duplicates().tolist()
        raise ValueError(
            f"Could not parse Boolean column {name}: {bad}"
        )

    return parsed.astype(bool)


def assert_equal_columns(
    frame: pd.DataFrame,
    left: str,
    right: str,
    *,
    numeric: bool = False,
    tolerance: float = 1e-10,
) -> None:
    if numeric:
        left_values = pd.to_numeric(frame[left], errors="coerce")
        right_values = pd.to_numeric(frame[right], errors="coerce")
        mismatch = ~np.isclose(
            left_values,
            right_values,
            rtol=0.0,
            atol=tolerance,
            equal_nan=True,
        )
    else:
        left_values = frame[left].fillna("").astype(str)
        right_values = frame[right].fillna("").astype(str)
        mismatch = left_values.ne(right_values)

    if mismatch.any():
        preview = frame.loc[
            mismatch,
            [
                "event_date",
                "market_id",
                "decision_rule",
                left,
                right,
            ],
        ].head(20)
        raise AssertionError(
            f"Input columns disagree: {left} versus {right}\n"
            + preview.to_string(index=False)
        )


def clipped_log_score(probability: pd.Series, outcome: pd.Series) -> pd.Series:
    clipped = probability.clip(
        LOG_EPSILON,
        1.0 - LOG_EPSILON,
    )
    return -(
        outcome * np.log(clipped)
        + (1 - outcome) * np.log(1 - clipped)
    )


def fmt_float(value: Any, digits: int = 6) -> str:
    if pd.isna(value):
        return ""
    return f"{float(value):.{digits}f}"

In [3]:
market_decision = pd.read_csv(
    MARKET_DECISION_PATH,
    dtype={
        "market_id": str,
        "condition_id": str,
        "selected_yes_token_id": str,
        "no_token_id": str,
    },
)
market_scoring = pd.read_csv(
    MARKET_SCORING_PATH,
    dtype={
        "market_id": str,
        "condition_id": str,
        "selected_yes_token_id": str,
        "no_token_id": str,
    },
)
weather_membership = pd.read_csv(
    WEATHER_MEMBERSHIP_PATH,
    dtype={
        "market_id": str,
        "condition_id": str,
        "yes_token_id": str,
        "no_token_id": str,
    },
)
weather_daily = pd.read_csv(WEATHER_DAILY_PATH)
outcomes = pd.read_csv(
    OUTCOME_PATH,
    dtype={
        "market_id": str,
        "condition_id": str,
        "yes_token_id": str,
        "no_token_id": str,
    },
)

with MARKET_SUMMARY_PATH.open(encoding="utf-8") as handle:
    market_summary_input = json.load(handle)

with WEATHER_SUMMARY_PATH.open(encoding="utf-8") as handle:
    weather_summary_input = json.load(handle)

for frame in (
    market_decision,
    market_scoring,
    weather_membership,
    weather_daily,
    outcomes,
):
    frame["event_date"] = pd.to_datetime(frame["event_date"])

market_decision["decision_cutoff_utc"] = pd.to_datetime(
    market_decision["decision_cutoff_utc"],
    utc=True,
)
market_decision["selected_price_timestamp_utc"] = pd.to_datetime(
    market_decision["selected_price_timestamp_utc"],
    utc=True,
    errors="coerce",
)
market_scoring["decision_cutoff_utc"] = pd.to_datetime(
    market_scoring["decision_cutoff_utc"],
    utc=True,
)
market_scoring["selected_price_timestamp_utc"] = pd.to_datetime(
    market_scoring["selected_price_timestamp_utc"],
    utc=True,
)
weather_membership["decision_cutoff_utc"] = pd.to_datetime(
    weather_membership["decision_cutoff_utc"],
    utc=True,
)
weather_membership["selected_run_initialisation_utc"] = pd.to_datetime(
    weather_membership["selected_run_initialisation_utc"],
    utc=True,
)
weather_daily["decision_cutoff_utc"] = pd.to_datetime(
    weather_daily["decision_cutoff_utc"],
    utc=True,
)
weather_daily["selected_run_initialisation_utc"] = pd.to_datetime(
    weather_daily["selected_run_initialisation_utc"],
    utc=True,
)
weather_daily["selected_run_available_utc"] = pd.to_datetime(
    weather_daily["selected_run_available_utc"],
    utc=True,
)

market_decision["price_available"] = parse_boolean(
    market_decision["price_available"],
    name="market_decision.price_available",
)
weather_membership["forecast_path_ready"] = parse_boolean(
    weather_membership["forecast_path_ready"],
    name="weather_membership.forecast_path_ready",
)
weather_daily["forecast_path_ready"] = parse_boolean(
    weather_daily["forecast_path_ready"],
    name="weather_daily.forecast_path_ready",
)

expected_rules = set(DECISION_RULE_ORDER)

if market_summary_input.get("verdict") != "PASS":
    raise AssertionError(
        "18p input is not a verified PASS release: "
        f"{market_summary_input.get('verdict')}"
    )

if weather_summary_input.get("verdict") not in {
    "PASS",
    "USABLE_WITH_LIMITATIONS",
}:
    raise AssertionError(
        "18q input is not usable: "
        f"{weather_summary_input.get('verdict')}"
    )

if len(market_decision) != 1320:
    raise AssertionError(
        f"Expected 1,320 market decision rows, found {len(market_decision)}"
    )
if len(market_scoring) != 1265:
    raise AssertionError(
        f"Expected 1,265 market scoring rows, found {len(market_scoring)}"
    )
if len(weather_membership) != 1320:
    raise AssertionError(
        "Expected 1,320 weather membership rows, "
        f"found {len(weather_membership)}"
    )
if len(weather_daily) != 120:
    raise AssertionError(
        f"Expected 120 weather daily rows, found {len(weather_daily)}"
    )
if len(outcomes) != 330:
    raise AssertionError(
        f"Expected 330 outcome rows, found {len(outcomes)}"
    )

key_columns = ["event_date", "market_id", "decision_rule"]

for name, frame in [
    ("market_decision", market_decision),
    ("market_scoring", market_scoring),
    ("weather_membership", weather_membership),
]:
    if frame.duplicated(key_columns).any():
        raise AssertionError(
            f"Duplicate exact join keys in {name}"
        )

if weather_daily.duplicated(
    ["event_date", "decision_rule"]
).any():
    raise AssertionError(
        "Duplicate date-rule keys in weather daily panel"
    )

if outcomes.duplicated(
    ["event_date", "market_id"]
).any():
    raise AssertionError(
        "Duplicate date-market keys in outcome panel"
    )

if market_decision["event_date"].nunique() != 30:
    raise AssertionError("18p does not contain 30 June dates")
if weather_membership["event_date"].nunique() != 30:
    raise AssertionError("18q does not contain 30 June dates")
if market_decision["market_id"].nunique() != 330:
    raise AssertionError("18p does not contain 330 markets")
if weather_membership["market_id"].nunique() != 330:
    raise AssertionError("18q does not contain 330 markets")
if set(market_decision["decision_rule"]) != expected_rules:
    raise AssertionError("Unexpected 18p decision-rule set")
if set(weather_membership["decision_rule"]) != expected_rules:
    raise AssertionError("Unexpected 18q decision-rule set")

scoring_keys = set(
    map(
        tuple,
        market_scoring[key_columns].itertuples(
            index=False,
            name=None,
        ),
    )
)
available_keys = set(
    map(
        tuple,
        market_decision.loc[
            market_decision["price_available"],
            key_columns,
        ].itertuples(index=False, name=None),
    )
)

if scoring_keys != available_keys:
    raise AssertionError(
        "18p scoring keys do not equal the available decision-price keys"
    )

print("Verified input validation: PASS")
print(f"Market decision rows: {len(market_decision)}")
print(f"Market scoring rows: {len(market_scoring)}")
print(f"Weather membership rows: {len(weather_membership)}")
print(f"Weather daily rows: {len(weather_daily)}")

Verified input validation: PASS
Market decision rows: 1320
Market scoring rows: 1265
Weather membership rows: 1320
Weather daily rows: 120


In [4]:
market_columns = [
    "event_date",
    "market_id",
    "decision_rule",
    "event_id",
    "event_slug",
    "condition_id",
    "market_slug",
    "question",
    "group_item_title",
    "event_type",
    "contract_event_type_v2",
    "selected_yes_token_id",
    "no_token_id",
    "hko_daily_max_c",
    "Y_event_int",
    "Y_no_int",
    "decision_rule_order",
    "decision_cutoff_hkt",
    "decision_cutoff_utc",
    "decision_cutoff_unix",
    "price_available",
    "p_market",
    "selected_price_timestamp_unix",
    "selected_price_timestamp_utc",
    "selected_price_timestamp_hkt",
    "price_staleness_hours",
    "selected_price_raw_response_path",
]

weather_columns = [
    "event_date",
    "market_id",
    "decision_rule",
    "event_id",
    "event_slug",
    "condition_id",
    "market_slug",
    "question",
    "group_item_title",
    "event_type",
    "canonical_label",
    "label_value_c",
    "lower_bound_c",
    "upper_bound_c",
    "yes_token_id",
    "no_token_id",
    "hko_daily_max_c",
    "realised_yes",
    "realised_no",
    "decision_rule_order",
    "decision_cutoff_utc",
    "selected_run_initialisation_utc",
    "selected_run_key",
    "forecast_path_ready",
    "forecast_daily_max_c",
    "forecast_error_c",
    "absolute_error_c",
    "deterministic_forecast_event_indicator",
]

support_audit = market_decision[
    market_columns
].merge(
    weather_membership[weather_columns],
    on=key_columns,
    how="outer",
    validate="one_to_one",
    suffixes=("_market", "_weather"),
    indicator=True,
)

if len(support_audit) != 1320:
    raise AssertionError(
        f"Exact outer join produced {len(support_audit)} rows, expected 1,320"
    )

if not support_audit["_merge"].eq("both").all():
    unmatched = support_audit.loc[
        ~support_audit["_merge"].eq("both")
    ]
    raise AssertionError(
        "The market and weather candidate universes differ:\n"
        + unmatched.head(20).to_string(index=False)
    )

consistency_pairs = [
    ("event_id_market", "event_id_weather", False),
    ("event_slug_market", "event_slug_weather", False),
    ("condition_id_market", "condition_id_weather", False),
    ("market_slug_market", "market_slug_weather", False),
    ("question_market", "question_weather", False),
    ("group_item_title_market", "canonical_label", False),
    ("event_type_market", "event_type_weather", False),
    ("selected_yes_token_id", "yes_token_id", False),
    ("no_token_id_market", "no_token_id_weather", False),
    ("hko_daily_max_c_market", "hko_daily_max_c_weather", True),
    ("Y_event_int", "realised_yes", True),
    ("Y_no_int", "realised_no", True),
    (
        "decision_rule_order_market",
        "decision_rule_order_weather",
        True,
    ),
]

for left, right, numeric in consistency_pairs:
    assert_equal_columns(
        support_audit,
        left,
        right,
        numeric=numeric,
    )

market_cutoff = pd.to_datetime(
    support_audit["decision_cutoff_utc_market"],
    utc=True,
)
weather_cutoff = pd.to_datetime(
    support_audit["decision_cutoff_utc_weather"],
    utc=True,
)
if not market_cutoff.eq(weather_cutoff).all():
    raise AssertionError(
        "Market and weather decision cut-offs disagree"
    )

support_audit["market_price_ready"] = parse_boolean(
    support_audit["price_available"],
    name="support_audit.price_available",
)
support_audit["weather_path_ready"] = parse_boolean(
    support_audit["forecast_path_ready"],
    name="support_audit.forecast_path_ready",
)
support_audit["common_support"] = (
    support_audit["market_price_ready"]
    & support_audit["weather_path_ready"]
)

support_audit["support_exclusion_reason"] = np.select(
    [
        (
            ~support_audit["market_price_ready"]
            & support_audit["weather_path_ready"]
        ),
        (
            support_audit["market_price_ready"]
            & ~support_audit["weather_path_ready"]
        ),
        (
            ~support_audit["market_price_ready"]
            & ~support_audit["weather_path_ready"]
        ),
    ],
    [
        "MARKET_PRICE_MISSING",
        "WEATHER_PATH_NOT_READY",
        "MARKET_PRICE_MISSING;WEATHER_PATH_NOT_READY",
    ],
    default="",
)

support_audit["event_date"] = pd.to_datetime(
    support_audit["event_date"]
)
support_audit["decision_rule_order"] = support_audit[
    "decision_rule"
].map(RULE_ORDER_MAP)

if support_audit["decision_rule_order"].isna().any():
    raise AssertionError("Unexpected decision rule after exact join")

no_lookahead_rows = support_audit.loc[
    support_audit["market_price_ready"]
    & (
        pd.to_datetime(
            support_audit["selected_price_timestamp_utc"],
            utc=True,
        )
        > pd.to_datetime(
            support_audit["decision_cutoff_utc_market"],
            utc=True,
        )
    )
]

if not no_lookahead_rows.empty:
    raise AssertionError(
        "At least one retained market observation is after its cut-off"
    )

weather_daily_check = weather_daily[
    [
        "event_date",
        "decision_rule",
        "selected_run_initialisation_utc",
        "selected_run_available_utc",
        "forecast_path_ready",
        "forecast_daily_max_c",
        "hko_daily_max_c",
        "forecast_error_c",
        "absolute_error_c",
    ]
].copy()

weather_daily_check["event_date"] = pd.to_datetime(
    weather_daily_check["event_date"]
)
weather_daily_check = weather_daily_check.rename(
    columns={
        "selected_run_initialisation_utc": (
            "daily_selected_run_initialisation_utc"
        ),
        "selected_run_available_utc": (
            "daily_selected_run_available_utc"
        ),
        "forecast_path_ready": "daily_forecast_path_ready",
        "forecast_daily_max_c": "daily_forecast_daily_max_c",
        "hko_daily_max_c": "daily_hko_daily_max_c",
        "forecast_error_c": "daily_forecast_error_c",
        "absolute_error_c": "daily_absolute_error_c",
    }
)

support_audit = support_audit.merge(
    weather_daily_check,
    on=["event_date", "decision_rule"],
    how="left",
    validate="many_to_one",
)

assert_equal_columns(
    support_audit,
    "selected_run_initialisation_utc",
    "daily_selected_run_initialisation_utc",
)
assert_equal_columns(
    support_audit,
    "forecast_daily_max_c",
    "daily_forecast_daily_max_c",
    numeric=True,
)
assert_equal_columns(
    support_audit,
    "hko_daily_max_c_weather",
    "daily_hko_daily_max_c",
    numeric=True,
)
assert_equal_columns(
    support_audit,
    "forecast_error_c",
    "daily_forecast_error_c",
    numeric=True,
)

if not (
    pd.to_datetime(
        support_audit["daily_selected_run_available_utc"],
        utc=True,
    )
    <= pd.to_datetime(
        support_audit["decision_cutoff_utc_market"],
        utc=True,
    )
).all():
    raise AssertionError(
        "At least one weather run was unavailable at its decision cut-off"
    )

print("Exact candidate-universe join: PASS")
print(f"Support-audit rows: {len(support_audit)}")
print(
    "Market-price-ready rows: "
    f"{int(support_audit['market_price_ready'].sum())}"
)
print(
    "Weather-path-ready rows: "
    f"{int(support_audit['weather_path_ready'].sum())}"
)
print(
    "Exact common-support rows: "
    f"{int(support_audit['common_support'].sum())}"
)

Exact candidate-universe join: PASS


Support-audit rows: 1320
Market-price-ready rows: 1265
Weather-path-ready rows: 1309
Exact common-support rows: 1254


In [5]:
scoring_columns = [
    "event_date",
    "market_id",
    "decision_rule",
    "brier_market",
    "log_score_market",
]

common_contract = support_audit.loc[
    support_audit["common_support"]
].copy()

common_contract = common_contract.merge(
    market_scoring[scoring_columns],
    on=key_columns,
    how="left",
    validate="one_to_one",
)

if common_contract[
    ["brier_market", "log_score_market"]
].isna().any().any():
    raise AssertionError(
        "A common-support row lacks upstream market scores"
    )

common_contract["market_binary_brier_recomputed"] = (
    common_contract["p_market"]
    - common_contract["Y_event_int"]
) ** 2
common_contract["market_binary_log_score_recomputed"] = (
    clipped_log_score(
        common_contract["p_market"],
        common_contract["Y_event_int"],
    )
)

if not np.isclose(
    common_contract["brier_market"],
    common_contract["market_binary_brier_recomputed"],
    rtol=0.0,
    atol=1e-12,
).all():
    raise AssertionError(
        "Recomputed market Brier scores disagree with 18p"
    )

if not np.isclose(
    common_contract["log_score_market"],
    common_contract["market_binary_log_score_recomputed"],
    rtol=0.0,
    atol=1e-12,
).all():
    raise AssertionError(
        "Recomputed market log scores disagree with 18p"
    )

common_contract["deterministic_forecast_event_indicator"] = (
    pd.to_numeric(
        common_contract[
            "deterministic_forecast_event_indicator"
        ],
        errors="raise",
    ).astype(int)
)
common_contract[
    "deterministic_event_indicator_squared_error"
] = (
    common_contract[
        "deterministic_forecast_event_indicator"
    ]
    - common_contract["Y_event_int"]
) ** 2

group_keys = ["event_date", "decision_rule"]

common_contract["n_contracts_in_common_book"] = (
    common_contract.groupby(group_keys)["market_id"].transform(
        "size"
    )
)
common_contract["market_book_probability_sum"] = (
    common_contract.groupby(group_keys)["p_market"].transform(
        "sum"
    )
)
common_contract["p_market_normalised"] = (
    common_contract["p_market"]
    / common_contract["market_book_probability_sum"]
)

book_size_counts = common_contract.groupby(
    group_keys
)["market_id"].size()

if not book_size_counts.eq(11).all():
    bad = book_size_counts.loc[~book_size_counts.eq(11)]
    raise AssertionError(
        "Exact common support contains incomplete event books:\n"
        + bad.to_string()
    )

actual_winner_counts = common_contract.groupby(
    group_keys
)["Y_event_int"].sum()
deterministic_winner_counts = common_contract.groupby(
    group_keys
)["deterministic_forecast_event_indicator"].sum()

if not actual_winner_counts.eq(1).all():
    raise AssertionError(
        "A common-support book lacks exactly one realised winner"
    )
if not deterministic_winner_counts.eq(1).all():
    raise AssertionError(
        "A common-support book lacks exactly one deterministic selected contract"
    )

if not common_contract["p_market"].between(0.0, 1.0).all():
    raise AssertionError(
        "A common-support market probability lies outside [0,1]"
    )
if not common_contract["p_market_normalised"].between(
    0.0,
    1.0,
).all():
    raise AssertionError(
        "A normalised market probability lies outside [0,1]"
    )

normalised_sums = common_contract.groupby(
    group_keys
)["p_market_normalised"].sum()

if not np.isclose(
    normalised_sums,
    1.0,
    rtol=0.0,
    atol=1e-12,
).all():
    raise AssertionError(
        "Normalised market probabilities do not sum to one"
    )

common_contract["market_binary_brier"] = common_contract[
    "market_binary_brier_recomputed"
]
common_contract["market_binary_log_score"] = common_contract[
    "market_binary_log_score_recomputed"
]

print("Common-support contract panel: PASS")
print(f"Contract-rule rows: {len(common_contract)}")
print(f"Complete date-rule books: {len(book_size_counts)}")

Common-support contract panel: PASS
Contract-rule rows: 1254
Complete date-rule books: 114


In [6]:
book_rows: list[dict[str, Any]] = []

for (event_date, decision_rule), book in common_contract.groupby(
    group_keys,
    sort=True,
):
    book = book.sort_values(
        ["label_value_c", "market_id"],
        na_position="first",
    ).copy()

    actual_winner = book.loc[
        book["Y_event_int"].eq(1)
    ]
    deterministic_selection = book.loc[
        book[
            "deterministic_forecast_event_indicator"
        ].eq(1)
    ]

    if len(actual_winner) != 1:
        raise AssertionError(
            f"Unexpected actual-winner count for {event_date}, {decision_rule}"
        )
    if len(deterministic_selection) != 1:
        raise AssertionError(
            "Unexpected deterministic-selection count for "
            f"{event_date}, {decision_rule}"
        )

    actual_winner = actual_winner.iloc[0]
    deterministic_selection = deterministic_selection.iloc[0]

    maximum_market_probability = float(
        book["p_market"].max()
    )
    market_modal_rows = book.loc[
        np.isclose(
            book["p_market"],
            maximum_market_probability,
            rtol=0.0,
            atol=1e-12,
        )
    ].sort_values("market_id").copy()

    n_market_modal_contracts = len(
        market_modal_rows
    )
    market_modal_tie = (
        n_market_modal_contracts > 1
    )
    market_modal_market_ids = "|".join(
        market_modal_rows["market_id"].astype(str)
    )
    market_modal_labels = "|".join(
        market_modal_rows["canonical_label"].astype(str)
    )
    market_modal_probability_normalised = float(
        market_modal_rows[
            "p_market_normalised"
        ].iloc[0]
    )
    market_modal_contains_actual_winner = int(
        market_modal_rows[
            "Y_event_int"
        ].eq(1).any()
    )
    market_modal_contains_deterministic = int(
        market_modal_rows[
            "market_id"
        ].eq(
            deterministic_selection[
                "market_id"
            ]
        ).any()
    )

    if n_market_modal_contracts == 1:
        unique_market_modal = (
            market_modal_rows.iloc[0]
        )
        unique_market_modal_market_id = (
            unique_market_modal["market_id"]
        )
        unique_market_modal_label = (
            unique_market_modal[
                "canonical_label"
            ]
        )
        unique_market_modal_exact_contract_hit = int(
            unique_market_modal[
                "Y_event_int"
            ]
            == 1
        )
        unique_market_modal_agrees_with_deterministic = int(
            unique_market_modal[
                "market_id"
            ]
            == deterministic_selection[
                "market_id"
            ]
        )
    else:
        unique_market_modal_market_id = ""
        unique_market_modal_label = ""
        unique_market_modal_exact_contract_hit = pd.NA
        unique_market_modal_agrees_with_deterministic = pd.NA

    total_probability = float(
        book["p_market"].sum()
    )
    normalised_probabilities = book[
        "p_market_normalised"
    ].to_numpy(dtype=float)
    outcomes_vector = book[
        "Y_event_int"
    ].to_numpy(dtype=float)

    winning_probability_raw = float(
        actual_winner["p_market"]
    )
    winning_probability_normalised = float(
        actual_winner["p_market_normalised"]
    )

    raw_categorical_log_score = -math.log(
        min(
            max(
                winning_probability_raw,
                LOG_EPSILON,
            ),
            1.0 - LOG_EPSILON,
        )
    )
    normalised_categorical_log_score = -math.log(
        min(
            max(
                winning_probability_normalised,
                LOG_EPSILON,
            ),
            1.0 - LOG_EPSILON,
        )
    )

    raw_multiclass_brier = float(
        np.square(
            book["p_market"].to_numpy(dtype=float)
            - outcomes_vector
        ).sum()
    )
    normalised_multiclass_brier = float(
        np.square(
            normalised_probabilities
            - outcomes_vector
        ).sum()
    )

    book_rows.append(
        {
            "event_date": event_date,
            "decision_rule": decision_rule,
            "decision_rule_order": RULE_ORDER_MAP[
                decision_rule
            ],
            "decision_cutoff_utc": actual_winner[
                "decision_cutoff_utc_market"
            ],
            "selected_run_initialisation_utc": actual_winner[
                "selected_run_initialisation_utc"
            ],
            "selected_run_available_utc": actual_winner[
                "daily_selected_run_available_utc"
            ],
            "selected_run_key": actual_winner[
                "selected_run_key"
            ],
            "n_contracts": len(book),
            "market_book_probability_sum": total_probability,
            "market_book_probability_error_vs_one": (
                total_probability - 1.0
            ),
            "max_market_price_staleness_hours": book[
                "price_staleness_hours"
            ].max(),
            "median_market_price_staleness_hours": book[
                "price_staleness_hours"
            ].median(),
            "hko_daily_max_c": float(
                actual_winner[
                    "hko_daily_max_c_market"
                ]
            ),
            "forecast_daily_max_c": float(
                actual_winner[
                    "forecast_daily_max_c"
                ]
            ),
            "forecast_error_c": float(
                actual_winner["forecast_error_c"]
            ),
            "absolute_error_c": float(
                actual_winner["absolute_error_c"]
            ),
            "actual_winning_market_id": actual_winner[
                "market_id"
            ],
            "actual_winning_label": actual_winner[
                "canonical_label"
            ],
            "actual_winner_market_probability_raw": (
                winning_probability_raw
            ),
            "actual_winner_market_probability_normalised": (
                winning_probability_normalised
            ),
            "deterministic_selected_market_id": (
                deterministic_selection["market_id"]
            ),
            "deterministic_selected_label": (
                deterministic_selection["canonical_label"]
            ),
            "deterministic_selected_market_probability_raw": float(
                deterministic_selection["p_market"]
            ),
            "deterministic_selected_market_probability_normalised": float(
                deterministic_selection[
                    "p_market_normalised"
                ]
            ),
            "deterministic_exact_contract_hit": int(
                deterministic_selection[
                    "Y_event_int"
                ]
                == 1
            ),
            "n_market_modal_contracts": (
                n_market_modal_contracts
            ),
            "market_modal_tie": (
                market_modal_tie
            ),
            "market_modal_market_ids": (
                market_modal_market_ids
            ),
            "market_modal_labels": (
                market_modal_labels
            ),
            "market_modal_probability_raw": (
                maximum_market_probability
            ),
            "market_modal_probability_normalised": (
                market_modal_probability_normalised
            ),
            "market_modal_contains_actual_winner": (
                market_modal_contains_actual_winner
            ),
            "market_modal_contains_deterministic": (
                market_modal_contains_deterministic
            ),
            "unique_market_modal_market_id": (
                unique_market_modal_market_id
            ),
            "unique_market_modal_label": (
                unique_market_modal_label
            ),
            "unique_market_modal_exact_contract_hit": (
                unique_market_modal_exact_contract_hit
            ),
            "unique_market_modal_agrees_with_deterministic": (
                unique_market_modal_agrees_with_deterministic
            ),
            "raw_categorical_log_score": (
                raw_categorical_log_score
            ),
            "normalised_categorical_log_score": (
                normalised_categorical_log_score
            ),
            "raw_multiclass_brier": (
                raw_multiclass_brier
            ),
            "normalised_multiclass_brier": (
                normalised_multiclass_brier
            ),
        }
    )

common_book = pd.DataFrame(book_rows).sort_values(
    ["event_date", "decision_rule_order"]
).reset_index(drop=True)

if len(common_book) != 114:
    raise AssertionError(
        f"Expected 114 exact common-support books, found {len(common_book)}"
    )

if not common_book["n_contracts"].eq(11).all():
    raise AssertionError(
        "A common-support book does not contain eleven contracts"
    )

if not (
    pd.to_datetime(
        common_book["selected_run_available_utc"],
        utc=True,
    )
    <= pd.to_datetime(
        common_book["decision_cutoff_utc"],
        utc=True,
    )
).all():
    raise AssertionError(
        "A retained book uses a run unavailable at its cut-off"
    )

print("Common-support book panel: PASS")
print(f"Book rows: {len(common_book)}")
display(common_book.head(8))

Common-support book panel: PASS
Book rows: 114


,event_date,decision_rule,decision_rule_order,decision_cutoff_utc,selected_run_initialisation_utc,selected_run_available_utc,selected_run_key,n_contracts,market_book_probability_sum,market_book_probability_error_vs_one,...,market_modal_contains_actual_winner,market_modal_contains_deterministic,unique_market_modal_market_id,unique_market_modal_label,unique_market_modal_exact_contract_hit,unique_market_modal_agrees_with_deterministic,raw_categorical_log_score,normalised_categorical_log_score,raw_multiclass_brier,normalised_multiclass_brier
0,2026-06-01,24h_prior,0,2026-05-30 16:00:00+00:00,2026-05-30 06:00:00+00:00,2026-05-30 12:00:00+00:00,20260530T0600Z,11,1.0275,0.0275,...,0,0,2391367,33°C or higher,0,0,1.203973,1.231101,0.701527,0.701661
1,2026-06-01,12h_prior,1,2026-05-31 04:00:00+00:00,2026-05-30 18:00:00+00:00,2026-05-31 00:00:00+00:00,20260530T1800Z,11,0.9825,-0.0175,...,0,0,2391367,33°C or higher,0,0,1.171183,1.153528,0.643561,0.641990
2,2026-06-01,6h_prior,2,2026-05-31 10:00:00+00:00,2026-05-31 00:00:00+00:00,2026-05-31 06:00:00+00:00,20260531T0000Z,11,0.9540,-0.0460,...,1,0,2391366,32°C,1,0,0.994252,0.947161,0.519595,0.509551
3,2026-06-01,event_day_open,3,2026-05-31 16:00:00+00:00,2026-05-31 06:00:00+00:00,2026-05-31 12:00:00+00:00,20260531T0600Z,11,1.0510,0.0510,...,1,0,2391366,32°C,1,0,1.021651,1.071393,0.573986,0.581085
4,2026-06-02,24h_prior,0,2026-05-31 16:00:00+00:00,2026-05-31 06:00:00+00:00,2026-05-31 12:00:00+00:00,20260531T0600Z,11,1.0135,0.0135,...,0,0,2399844,32°C,0,0,1.255266,1.268676,0.666388,0.667725
5,2026-06-02,12h_prior,1,2026-06-01 04:00:00+00:00,2026-05-31 18:00:00+00:00,2026-06-01 00:00:00+00:00,20260531T1800Z,11,1.0340,0.0340,...,0,0,2399844,32°C,0,0,1.187444,1.220878,0.662881,0.665288
6,2026-06-02,6h_prior,2,2026-06-01 10:00:00+00:00,2026-06-01 00:00:00+00:00,2026-06-01 06:00:00+00:00,20260601T0000Z,11,1.0435,0.0435,...,1,0,2399845,33°C,1,0,0.693147,0.735728,0.357062,0.369600
7,2026-06-02,event_day_open,3,2026-06-01 16:00:00+00:00,2026-06-01 06:00:00+00:00,2026-06-01 12:00:00+00:00,20260601T0600Z,11,0.9385,-0.0615,...,1,0,2399845,33°C,1,0,0.994252,0.930780,0.526226,0.513769


In [7]:
support_flow_rows: list[dict[str, Any]] = []

for decision_rule in DECISION_RULE_ORDER:
    subset = support_audit.loc[
        support_audit["decision_rule"].eq(decision_rule)
    ]

    common_subset = subset.loc[
        subset["common_support"]
    ]

    support_flow_rows.append(
        {
            "decision_rule": decision_rule,
            "decision_rule_order": RULE_ORDER_MAP[
                decision_rule
            ],
            "candidate_contract_rows": len(subset),
            "market_price_ready_rows": int(
                subset["market_price_ready"].sum()
            ),
            "weather_path_ready_rows": int(
                subset["weather_path_ready"].sum()
            ),
            "common_support_contract_rows": int(
                subset["common_support"].sum()
            ),
            "market_price_missing_rows": int(
                (
                    ~subset["market_price_ready"]
                    & subset["weather_path_ready"]
                ).sum()
            ),
            "weather_path_not_ready_rows": int(
                (
                    subset["market_price_ready"]
                    & ~subset["weather_path_ready"]
                ).sum()
            ),
            "both_missing_rows": int(
                (
                    ~subset["market_price_ready"]
                    & ~subset["weather_path_ready"]
                ).sum()
            ),
            "candidate_date_rule_books": int(
                subset["event_date"].nunique()
            ),
            "common_support_books": int(
                common_subset["event_date"].nunique()
            ),
            "common_support_dates": "|".join(
                sorted(
                    common_subset[
                        "event_date"
                    ].dt.strftime("%Y-%m-%d").unique()
                )
            ),
            "excluded_dates": "|".join(
                sorted(
                    subset.loc[
                        ~subset["common_support"],
                        "event_date",
                    ].dt.strftime("%Y-%m-%d").unique()
                )
            ),
        }
    )

support_flow = pd.DataFrame(support_flow_rows)

market_binary_rows: list[dict[str, Any]] = []

for decision_rule in DECISION_RULE_ORDER:
    rule_frame = common_contract.loc[
        common_contract["decision_rule"].eq(decision_rule)
    ]

    event_type_subsets = [
        ("ALL", rule_frame),
        (
            "interior_bin",
            rule_frame.loc[
                rule_frame[
                    "contract_event_type_v2"
                ].eq("interior_bin")
            ],
        ),
        (
            "lower_tail_endpoint",
            rule_frame.loc[
                rule_frame[
                    "contract_event_type_v2"
                ].eq("lower_tail_endpoint")
            ],
        ),
        (
            "upper_tail",
            rule_frame.loc[
                rule_frame[
                    "contract_event_type_v2"
                ].eq("upper_tail")
            ],
        ),
    ]

    for event_type, subset in event_type_subsets:
        if subset.empty:
            continue

        market_binary_rows.append(
            {
                "decision_rule": decision_rule,
                "decision_rule_order": RULE_ORDER_MAP[
                    decision_rule
                ],
                "contract_event_type": event_type,
                "n": len(subset),
                "n_dates": subset[
                    "event_date"
                ].nunique(),
                "mean_brier": subset[
                    "market_binary_brier"
                ].mean(),
                "median_brier": subset[
                    "market_binary_brier"
                ].median(),
                "mean_log_score": subset[
                    "market_binary_log_score"
                ].mean(),
                "median_log_score": subset[
                    "market_binary_log_score"
                ].median(),
                "mean_market_probability": subset[
                    "p_market"
                ].mean(),
                "outcome_rate": subset[
                    "Y_event_int"
                ].mean(),
                "median_price_staleness_hours": subset[
                    "price_staleness_hours"
                ].median(),
                "p95_price_staleness_hours": subset[
                    "price_staleness_hours"
                ].quantile(0.95),
            }
        )

market_binary_summary = pd.DataFrame(
    market_binary_rows
).sort_values(
    [
        "decision_rule_order",
        "contract_event_type",
    ]
)

categorical_rows: list[dict[str, Any]] = []

for decision_rule in DECISION_RULE_ORDER:
    subset = common_book.loc[
        common_book["decision_rule"].eq(decision_rule)
    ]

    categorical_rows.append(
        {
            "decision_rule": decision_rule,
            "decision_rule_order": RULE_ORDER_MAP[
                decision_rule
            ],
            "n_complete_books": len(subset),
            "mean_book_probability_sum": subset[
                "market_book_probability_sum"
            ].mean(),
            "median_book_probability_sum": subset[
                "market_book_probability_sum"
            ].median(),
            "mean_abs_probability_sum_error": subset[
                "market_book_probability_error_vs_one"
            ].abs().mean(),
            "mean_raw_categorical_log_score": subset[
                "raw_categorical_log_score"
            ].mean(),
            "mean_normalised_categorical_log_score": subset[
                "normalised_categorical_log_score"
            ].mean(),
            "mean_raw_multiclass_brier": subset[
                "raw_multiclass_brier"
            ].mean(),
            "mean_normalised_multiclass_brier": subset[
                "normalised_multiclass_brier"
            ].mean(),
            "market_modal_tied_books": int(
                subset["market_modal_tie"].sum()
            ),
            "market_modal_tie_rate": subset[
                "market_modal_tie"
            ].mean(),
            "market_modal_contains_actual_winner_rate": subset[
                "market_modal_contains_actual_winner"
            ].mean(),
            "unique_market_modal_books": int(
                (~subset["market_modal_tie"]).sum()
            ),
            "unique_market_modal_exact_contract_hit_rate": pd.to_numeric(
                subset[
                    "unique_market_modal_exact_contract_hit"
                ],
                errors="coerce",
            ).mean(),
            "deterministic_exact_contract_hit_rate": subset[
                "deterministic_exact_contract_hit"
            ].mean(),
            "market_modal_contains_deterministic_rate": subset[
                "market_modal_contains_deterministic"
            ].mean(),
            "unique_market_modal_deterministic_agreement_rate": pd.to_numeric(
                subset[
                    "unique_market_modal_agrees_with_deterministic"
                ],
                errors="coerce",
            ).mean(),
            "mean_actual_winner_probability_normalised": subset[
                "actual_winner_market_probability_normalised"
            ].mean(),
            "mean_deterministic_selected_probability_normalised": subset[
                "deterministic_selected_market_probability_normalised"
            ].mean(),
        }
    )

market_categorical_summary = pd.DataFrame(
    categorical_rows
)

weather_rows: list[dict[str, Any]] = []

for decision_rule in DECISION_RULE_ORDER:
    subset = common_book.loc[
        common_book["decision_rule"].eq(decision_rule)
    ]

    weather_rows.append(
        {
            "decision_rule": decision_rule,
            "decision_rule_order": RULE_ORDER_MAP[
                decision_rule
            ],
            "n_common_support_dates": len(subset),
            "mean_forecast_daily_max_c": subset[
                "forecast_daily_max_c"
            ].mean(),
            "mean_hko_daily_max_c": subset[
                "hko_daily_max_c"
            ].mean(),
            "mean_error_c": subset[
                "forecast_error_c"
            ].mean(),
            "mae_c": subset[
                "absolute_error_c"
            ].mean(),
            "rmse_c": math.sqrt(
                np.square(
                    subset["forecast_error_c"]
                ).mean()
            ),
            "median_absolute_error_c": subset[
                "absolute_error_c"
            ].median(),
            "underforecast_rate": (
                subset["forecast_error_c"] < 0
            ).mean(),
            "deterministic_exact_contract_hit_rate": subset[
                "deterministic_exact_contract_hit"
            ].mean(),
        }
    )

weather_error_summary = pd.DataFrame(weather_rows)

support_exclusions = support_audit.loc[
    ~support_audit["common_support"]
].copy()

print("Support flow:")
display(support_flow)
print("Market scores on exact support:")
display(
    market_binary_summary.loc[
        market_binary_summary[
            "contract_event_type"
        ].eq("ALL")
    ]
)
print("Categorical and deterministic descriptive summary:")
display(market_categorical_summary)
print("Weather errors on exact support:")
display(weather_error_summary)

Support flow:


,decision_rule,decision_rule_order,candidate_contract_rows,market_price_ready_rows,weather_path_ready_rows,common_support_contract_rows,market_price_missing_rows,weather_path_not_ready_rows,both_missing_rows,candidate_date_rule_books,common_support_books,common_support_dates,excluded_dates
0,24h_prior,0,330,297,330,297,33,0,0,30,27,2026-06-01|2026-06-02|2026-06-03|2026-06-04|20...,2026-06-06|2026-06-07|2026-06-08
1,12h_prior,1,330,308,330,308,22,0,0,30,28,2026-06-01|2026-06-02|2026-06-03|2026-06-04|20...,2026-06-06|2026-06-07
2,6h_prior,2,330,330,319,319,0,11,0,30,29,2026-06-01|2026-06-02|2026-06-03|2026-06-04|20...,2026-06-24
3,event_day_open,3,330,330,330,330,0,0,0,30,30,2026-06-01|2026-06-02|2026-06-03|2026-06-04|20...,


Market scores on exact support:


,decision_rule,decision_rule_order,contract_event_type,n,n_dates,mean_brier,median_brier,mean_log_score,median_log_score,mean_market_probability,outcome_rate,median_price_staleness_hours,p95_price_staleness_hours
0,24h_prior,0,ALL,297,27,0.058715,0.000380,0.184025,0.019693,0.094088,0.090909,0.998056,0.999167
4,12h_prior,1,ALL,308,28,0.058737,0.000298,0.184717,0.017401,0.093131,0.090909,0.998333,0.999167
8,6h_prior,2,ALL,319,29,0.056012,0.000182,0.177429,0.013592,0.093580,0.090909,0.998333,0.999167
12,event_day_open,3,ALL,330,30,0.055751,0.000086,0.177376,0.009293,0.094024,0.090909,0.998333,0.999167


Categorical and deterministic descriptive summary:


,decision_rule,decision_rule_order,n_complete_books,mean_book_probability_sum,median_book_probability_sum,mean_abs_probability_sum_error,mean_raw_categorical_log_score,mean_normalised_categorical_log_score,mean_raw_multiclass_brier,mean_normalised_multiclass_brier,market_modal_tied_books,market_modal_tie_rate,market_modal_contains_actual_winner_rate,unique_market_modal_books,unique_market_modal_exact_contract_hit_rate,deterministic_exact_contract_hit_rate,market_modal_contains_deterministic_rate,unique_market_modal_deterministic_agreement_rate,mean_actual_winner_probability_normalised,mean_deterministic_selected_probability_normalised
0,24h_prior,0,27,1.034963,1.03750,0.043333,1.195824,1.229658,0.645870,0.648074,1,0.037037,0.518519,26,0.500000,0.000000,0.037037,0.038462,0.305920,0.104847
1,12h_prior,1,28,1.024446,1.03625,0.038518,1.215376,1.239007,0.646106,0.648440,0,0.000000,0.500000,28,0.500000,0.071429,0.035714,0.035714,0.311918,0.115447
2,6h_prior,2,29,1.029379,1.03750,0.039517,1.158902,1.187355,0.616133,0.617830,0,0.000000,0.551724,29,0.551724,0.103448,0.000000,0.000000,0.335354,0.113282
3,event_day_open,3,30,1.034267,1.04875,0.049000,1.156844,1.189793,0.613258,0.613239,1,0.033333,0.633333,29,0.620690,0.066667,0.066667,0.034483,0.345413,0.111799


Weather errors on exact support:


,decision_rule,decision_rule_order,n_common_support_dates,mean_forecast_daily_max_c,mean_hko_daily_max_c,mean_error_c,mae_c,rmse_c,median_absolute_error_c,underforecast_rate,deterministic_exact_contract_hit_rate
0,24h_prior,0,27,29.407407,31.181481,-1.774074,1.833333,2.016139,1.70,0.962963,0.000000
1,12h_prior,1,28,29.425000,31.171429,-1.746429,1.746429,1.889539,1.65,1.000000,0.071429
2,6h_prior,2,29,29.462069,31.117241,-1.655172,1.758621,1.978854,1.70,0.931034,0.103448
3,event_day_open,3,30,29.516667,31.173333,-1.656667,1.750000,2.011384,1.55,0.966667,0.066667


In [8]:
integrity_rows: list[dict[str, Any]] = []

def add_check(
    check: str,
    passed: bool,
    detail: str,
    blocking: bool = True,
) -> None:
    integrity_rows.append(
        {
            "check": check,
            "passed": bool(passed),
            "detail": detail,
            "blocking": bool(blocking),
        }
    )

add_check(
    "candidate_universe_has_1320_rows",
    len(support_audit) == 1320,
    f"rows={len(support_audit)}",
)
add_check(
    "candidate_universe_has_unique_exact_keys",
    not support_audit.duplicated(key_columns).any(),
    (
        "duplicates="
        f"{int(support_audit.duplicated(key_columns).sum())}"
    ),
)
add_check(
    "market_price_ready_rows_equal_1265",
    int(support_audit["market_price_ready"].sum()) == 1265,
    (
        "ready="
        f"{int(support_audit['market_price_ready'].sum())}"
    ),
)
add_check(
    "weather_path_ready_rows_equal_1309",
    int(support_audit["weather_path_ready"].sum()) == 1309,
    (
        "ready="
        f"{int(support_audit['weather_path_ready'].sum())}"
    ),
)
add_check(
    "exact_common_support_has_1254_rows",
    len(common_contract) == 1254,
    f"rows={len(common_contract)}",
)
add_check(
    "support_exclusions_have_66_rows",
    len(support_exclusions) == 66,
    f"rows={len(support_exclusions)}",
)
add_check(
    "common_support_has_114_complete_books",
    len(common_book) == 114,
    f"books={len(common_book)}",
)
add_check(
    "all_common_support_books_have_11_contracts",
    common_book["n_contracts"].eq(11).all(),
    (
        "bad_books="
        f"{int((~common_book['n_contracts'].eq(11)).sum())}"
    ),
)
add_check(
    "all_common_support_books_have_one_actual_winner",
    common_contract.groupby(
        group_keys
    )["Y_event_int"].sum().eq(1).all(),
    "exactly one realised winner per retained book",
)
add_check(
    "all_common_support_books_have_one_deterministic_selection",
    common_contract.groupby(
        group_keys
    )[
        "deterministic_forecast_event_indicator"
    ].sum().eq(1).all(),
    "exactly one deterministic selected contract per retained book",
)
add_check(
    "normalised_market_probabilities_sum_to_one",
    np.isclose(
        common_contract.groupby(
            group_keys
        )["p_market_normalised"].sum(),
        1.0,
        rtol=0.0,
        atol=1e-12,
    ).all(),
    "complete-book normalisation only",
)
add_check(
    "market_observations_are_no_lookahead",
    no_lookahead_rows.empty,
    f"violations={len(no_lookahead_rows)}",
)
add_check(
    "weather_runs_are_admissible_at_cutoff",
    (
        pd.to_datetime(
            common_book[
                "selected_run_available_utc"
            ],
            utc=True,
        )
        <= pd.to_datetime(
            common_book[
                "decision_cutoff_utc"
            ],
            utc=True,
        )
    ).all(),
    "selected run availability does not exceed decision time",
)
add_check(
    "market_scores_recompute_exactly",
    (
        np.isclose(
            common_contract[
                "brier_market"
            ],
            common_contract[
                "market_binary_brier"
            ],
            rtol=0.0,
            atol=1e-12,
        ).all()
        and np.isclose(
            common_contract[
                "log_score_market"
            ],
            common_contract[
                "market_binary_log_score"
            ],
            rtol=0.0,
            atol=1e-12,
        ).all()
    ),
    "18p scores equal independent recomputation",
)
add_check(
    "market_modal_sets_are_nonempty",
    common_book[
        "n_market_modal_contracts"
    ].ge(1).all(),
    (
        "bad_books="
        f"{int((~common_book['n_market_modal_contracts'].ge(1)).sum())}"
    ),
)
add_check(
    "market_modal_tie_flag_matches_modal_count",
    (
        common_book[
            "market_modal_tie"
        ]
        == common_book[
            "n_market_modal_contracts"
        ].gt(1)
    ).all(),
    (
        "mismatches="
        f"{int((common_book['market_modal_tie'] != common_book['n_market_modal_contracts'].gt(1)).sum())}"
    ),
)
add_check(
    "tie_aware_market_modal_hit_is_binary",
    common_book[
        "market_modal_contains_actual_winner"
    ].isin([0, 1]).all(),
    "tie-aware modal hit indicator",
)
add_check(
    "tie_aware_market_modal_deterministic_overlap_is_binary",
    common_book[
        "market_modal_contains_deterministic"
    ].isin([0, 1]).all(),
    "tie-aware modal-deterministic overlap indicator",
)
add_check(
    "no_gaussian_bridge_probability_columns",
    not any(
        (
            "gaussian" in column.lower()
            or column.lower().startswith(
                "p_ecmwf_proxy"
            )
        )
        for column in common_contract.columns
    ),
    "deterministic weather remains a point forecast",
)

actual_common_rows = (
    support_flow.set_index("decision_rule")[
        "common_support_contract_rows"
    ].to_dict()
)
actual_common_books = (
    support_flow.set_index("decision_rule")[
        "common_support_books"
    ].to_dict()
)

add_check(
    "rule_level_common_rows_match_expected",
    all(
        int(actual_common_rows[rule])
        == EXPECTED_COMMON_ROWS[rule]
        for rule in DECISION_RULE_ORDER
    ),
    json.dumps(
        {
            rule: int(actual_common_rows[rule])
            for rule in DECISION_RULE_ORDER
        },
        sort_keys=True,
    ),
)
add_check(
    "rule_level_common_books_match_expected",
    all(
        int(actual_common_books[rule])
        == EXPECTED_COMMON_BOOKS[rule]
        for rule in DECISION_RULE_ORDER
    ),
    json.dumps(
        {
            rule: int(actual_common_books[rule])
            for rule in DECISION_RULE_ORDER
        },
        sort_keys=True,
    ),
)

exclusion_counts = support_exclusions[
    "support_exclusion_reason"
].value_counts().to_dict()

add_check(
    "market_only_exclusions_equal_55",
    int(
        exclusion_counts.get(
            "MARKET_PRICE_MISSING",
            0,
        )
    )
    == 55,
    json.dumps(exclusion_counts, sort_keys=True),
)
add_check(
    "weather_only_exclusions_equal_11",
    int(
        exclusion_counts.get(
            "WEATHER_PATH_NOT_READY",
            0,
        )
    )
    == 11,
    json.dumps(exclusion_counts, sort_keys=True),
)
add_check(
    "no_joint_market_weather_missing_rows",
    int(
        exclusion_counts.get(
            "MARKET_PRICE_MISSING;WEATHER_PATH_NOT_READY",
            0,
        )
    )
    == 0,
    json.dumps(exclusion_counts, sort_keys=True),
)

integrity_checks = pd.DataFrame(integrity_rows)

blocking_failures = integrity_checks.loc[
    integrity_checks["blocking"]
    & ~integrity_checks["passed"]
]

if not blocking_failures.empty:
    raise AssertionError(
        "Blocking integrity checks failed:\n"
        + blocking_failures.to_string(index=False)
    )

issue_columns = [
    "issue_level",
    "issue_code",
    "event_date",
    "market_id",
    "decision_rule",
    "detail",
    "blocking",
]
issues = pd.DataFrame(columns=issue_columns)

verdict = "PASS"

print(f"Final verdict: {verdict}")
display(integrity_checks)

Final verdict: PASS


,check,passed,detail,blocking
0,candidate_universe_has_1320_rows,True,rows=1320,True
1,candidate_universe_has_unique_exact_keys,True,duplicates=0,True
2,market_price_ready_rows_equal_1265,True,ready=1265,True
3,weather_path_ready_rows_equal_1309,True,ready=1309,True
4,exact_common_support_has_1254_rows,True,rows=1254,True
5,support_exclusions_have_66_rows,True,rows=66,True
6,common_support_has_114_complete_books,True,books=114,True
7,all_common_support_books_have_11_contracts,True,bad_books=0,True
8,all_common_support_books_have_one_actual_winner,True,exactly one realised winner per retained book,True
9,all_common_support_books_have_one_deterministi...,True,exactly one deterministic selected contract pe...,True


In [9]:
# Prepare stable, concise canonical columns.
support_audit_output = support_audit[
    [
        "event_date",
        "event_id_market",
        "event_slug_market",
        "market_id",
        "condition_id_market",
        "market_slug_market",
        "question_market",
        "canonical_label",
        "event_type_market",
        "contract_event_type_v2",
        "label_value_c",
        "lower_bound_c",
        "upper_bound_c",
        "selected_yes_token_id",
        "no_token_id_market",
        "hko_daily_max_c_market",
        "Y_event_int",
        "Y_no_int",
        "decision_rule",
        "decision_rule_order",
        "decision_cutoff_hkt",
        "decision_cutoff_utc_market",
        "market_price_ready",
        "p_market",
        "selected_price_timestamp_utc",
        "selected_price_timestamp_hkt",
        "price_staleness_hours",
        "weather_path_ready",
        "selected_run_initialisation_utc",
        "daily_selected_run_available_utc",
        "selected_run_key",
        "forecast_daily_max_c",
        "forecast_error_c",
        "absolute_error_c",
        "deterministic_forecast_event_indicator",
        "common_support",
        "support_exclusion_reason",
    ]
].copy()

support_audit_output = support_audit_output.rename(
    columns={
        "event_id_market": "event_id",
        "event_slug_market": "event_slug",
        "condition_id_market": "condition_id",
        "market_slug_market": "market_slug",
        "question_market": "question",
        "event_type_market": "event_type",
        "no_token_id_market": "no_token_id",
        "hko_daily_max_c_market": "hko_daily_max_c",
        "decision_cutoff_utc_market": "decision_cutoff_utc",
        "daily_selected_run_available_utc": (
            "selected_run_available_utc"
        ),
    }
)

common_contract_output = common_contract[
    [
        "event_date",
        "event_id_market",
        "event_slug_market",
        "market_id",
        "condition_id_market",
        "market_slug_market",
        "question_market",
        "canonical_label",
        "event_type_market",
        "contract_event_type_v2",
        "label_value_c",
        "lower_bound_c",
        "upper_bound_c",
        "selected_yes_token_id",
        "no_token_id_market",
        "hko_daily_max_c_market",
        "Y_event_int",
        "Y_no_int",
        "decision_rule",
        "decision_rule_order",
        "decision_cutoff_hkt",
        "decision_cutoff_utc_market",
        "p_market",
        "market_book_probability_sum",
        "p_market_normalised",
        "selected_price_timestamp_utc",
        "selected_price_timestamp_hkt",
        "price_staleness_hours",
        "selected_run_initialisation_utc",
        "daily_selected_run_available_utc",
        "selected_run_key",
        "forecast_daily_max_c",
        "forecast_error_c",
        "absolute_error_c",
        "deterministic_forecast_event_indicator",
        "market_binary_brier",
        "market_binary_log_score",
        "deterministic_event_indicator_squared_error",
    ]
].copy()

common_contract_output = common_contract_output.rename(
    columns={
        "event_id_market": "event_id",
        "event_slug_market": "event_slug",
        "condition_id_market": "condition_id",
        "market_slug_market": "market_slug",
        "question_market": "question",
        "event_type_market": "event_type",
        "no_token_id_market": "no_token_id",
        "hko_daily_max_c_market": "hko_daily_max_c",
        "decision_cutoff_utc_market": "decision_cutoff_utc",
        "daily_selected_run_available_utc": (
            "selected_run_available_utc"
        ),
    }
)

support_exclusions_output = support_audit_output.loc[
    ~support_audit_output["common_support"]
].copy()

frames_with_dates = [
    support_audit_output,
    common_contract_output,
    common_book,
    support_exclusions_output,
]

for frame in frames_with_dates:
    frame["event_date"] = pd.to_datetime(
        frame["event_date"]
    ).dt.date.astype(str)

time_columns_by_frame = [
    (
        support_audit_output,
        [
            "decision_cutoff_utc",
            "selected_price_timestamp_utc",
            "selected_run_initialisation_utc",
            "selected_run_available_utc",
        ],
    ),
    (
        common_contract_output,
        [
            "decision_cutoff_utc",
            "selected_price_timestamp_utc",
            "selected_run_initialisation_utc",
            "selected_run_available_utc",
        ],
    ),
    (
        common_book,
        [
            "decision_cutoff_utc",
            "selected_run_initialisation_utc",
            "selected_run_available_utc",
        ],
    ),
    (
        support_exclusions_output,
        [
            "decision_cutoff_utc",
            "selected_price_timestamp_utc",
            "selected_run_initialisation_utc",
            "selected_run_available_utc",
        ],
    ),
]

for frame, columns in time_columns_by_frame:
    for column in columns:
        if column in frame.columns:
            frame[column] = frame[column].astype(str)

output_paths = {
    "support_audit": (
        OUT_DIR
        / "18r_june_2026_support_audit_panel.csv"
    ),
    "common_contract": (
        OUT_DIR
        / "18r_june_2026_common_support_contract_panel.csv"
    ),
    "common_book": (
        OUT_DIR
        / "18r_june_2026_common_support_book_panel.csv"
    ),
    "support_exclusions": (
        OUT_DIR
        / "18r_june_2026_support_exclusions.csv"
    ),
    "support_flow": (
        OUT_DIR
        / "18r_june_2026_support_flow_summary.csv"
    ),
    "market_binary": (
        OUT_DIR
        / "18r_june_2026_market_binary_score_summary.csv"
    ),
    "market_categorical": (
        OUT_DIR
        / "18r_june_2026_market_categorical_score_summary.csv"
    ),
    "weather_error": (
        OUT_DIR
        / "18r_june_2026_weather_error_summary.csv"
    ),
    "integrity": (
        OUT_DIR
        / "18r_june_2026_integrity_checks.csv"
    ),
    "issues": (
        OUT_DIR
        / "18r_june_2026_common_support_issues.csv"
    ),
}

support_audit_output.to_csv(
    output_paths["support_audit"],
    index=False,
)
common_contract_output.to_csv(
    output_paths["common_contract"],
    index=False,
)
common_book.to_csv(
    output_paths["common_book"],
    index=False,
)
support_exclusions_output.to_csv(
    output_paths["support_exclusions"],
    index=False,
)
support_flow.to_csv(
    output_paths["support_flow"],
    index=False,
)
market_binary_summary.to_csv(
    output_paths["market_binary"],
    index=False,
)
market_categorical_summary.to_csv(
    output_paths["market_categorical"],
    index=False,
)
weather_error_summary.to_csv(
    output_paths["weather_error"],
    index=False,
)
integrity_checks.to_csv(
    output_paths["integrity"],
    index=False,
)
issues.to_csv(
    output_paths["issues"],
    index=False,
)

summary = {
    "step": STEP,
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "verdict": verdict,
    "join_key": [
        "event_date",
        "market_id",
        "decision_rule",
    ],
    "candidate_contract_rule_rows": int(
        len(support_audit_output)
    ),
    "market_price_ready_rows": int(
        support_audit_output[
            "market_price_ready"
        ].sum()
    ),
    "weather_path_ready_rows": int(
        support_audit_output[
            "weather_path_ready"
        ].sum()
    ),
    "common_support_contract_rule_rows": int(
        len(common_contract_output)
    ),
    "excluded_contract_rule_rows": int(
        len(support_exclusions_output)
    ),
    "common_support_complete_books": int(
        len(common_book)
    ),
    "market_modal_tied_books": int(
        common_book[
            "market_modal_tie"
        ].sum()
    ),
    "unique_market_modal_books": int(
        (~common_book[
            "market_modal_tie"
        ]).sum()
    ),
    "market_price_missing_rows": int(
        (
            support_exclusions_output[
                "support_exclusion_reason"
            ]
            == "MARKET_PRICE_MISSING"
        ).sum()
    ),
    "weather_path_not_ready_rows": int(
        (
            support_exclusions_output[
                "support_exclusion_reason"
            ]
            == "WEATHER_PATH_NOT_READY"
        ).sum()
    ),
    "joint_missing_rows": int(
        (
            support_exclusions_output[
                "support_exclusion_reason"
            ]
            == (
                "MARKET_PRICE_MISSING;"
                "WEATHER_PATH_NOT_READY"
            )
        ).sum()
    ),
    "common_support_rows_by_rule": {
        row.decision_rule: int(
            row.common_support_contract_rows
        )
        for row in support_flow.itertuples(
            index=False
        )
    },
    "common_support_books_by_rule": {
        row.decision_rule: int(
            row.common_support_books
        )
        for row in support_flow.itertuples(
            index=False
        )
    },
    "probability_bridge_applied": False,
    "artificial_ensemble_features_created": False,
    "market_probability_normalisation": (
        "Applied only within complete eleven-contract "
        "common-support books for distributional diagnostics."
    ),
    "deterministic_weather_role": (
        "Point forecast and future local residual-postprocessing input; "
        "not treated as a calibrated probability."
    ),
    "integrity_checks_passed": int(
        integrity_checks["passed"].sum()
    ),
    "integrity_checks_total": int(
        len(integrity_checks)
    ),
    "issue_rows": int(len(issues)),
}

summary_path = (
    OUT_DIR
    / "18r_june_2026_common_support_summary.json"
)
summary_path.write_text(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

environment = {
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "market_input": str(
        MARKET_DECISION_PATH.relative_to(
            REPO_ROOT
        )
    ),
    "weather_input": str(
        WEATHER_MEMBERSHIP_PATH.relative_to(
            REPO_ROOT
        )
    ),
    "outcome_input": str(
        OUTCOME_PATH.relative_to(
            REPO_ROOT
        )
    ),
    "log_epsilon": LOG_EPSILON,
    "probability_bridge_applied": False,
}

environment_path = (
    OUT_DIR
    / "18r_june_2026_environment.json"
)
environment_path.write_text(
    json.dumps(
        environment,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

print(json.dumps(summary, indent=2, ensure_ascii=False))

{
  "step": "18r",
  "generated_at_utc": "2026-07-21T17:24:18.866329+00:00",
  "verdict": "PASS",
  "join_key": [
    "event_date",
    "market_id",
    "decision_rule"
  ],
  "candidate_contract_rule_rows": 1320,
  "market_price_ready_rows": 1265,
  "weather_path_ready_rows": 1309,
  "common_support_contract_rule_rows": 1254,
  "excluded_contract_rule_rows": 66,
  "common_support_complete_books": 114,
  "market_modal_tied_books": 2,
  "unique_market_modal_books": 112,
  "market_price_missing_rows": 55,
  "weather_path_not_ready_rows": 11,
  "joint_missing_rows": 0,
  "common_support_rows_by_rule": {
    "24h_prior": 297,
    "12h_prior": 308,
    "6h_prior": 319,
    "event_day_open": 330
  },
  "common_support_books_by_rule": {
    "24h_prior": 27,
    "12h_prior": 28,
    "6h_prior": 29,
    "event_day_open": 30
  },
  "probability_bridge_applied": false,
  "artificial_ensemble_features_created": false,
  "market_probability_normalisation": "Applied only within complete eleven-contr

In [10]:
report_lines = [
    "# 18r June 2026 market–weather exact common support",
    "",
    f"Generated at UTC: `{summary['generated_at_utc']}`",
    "",
    "## Overall judgement",
    "",
    f"**{summary['verdict']}**",
    "",
    "## Exact support definition",
    "",
    (
        "A contract-rule row is retained only when a verified "
        "Polymarket price is available at or before the decision "
        "cut-off and the selected issue-time-admissible ECMWF run "
        "contains all 24 non-missing Hong Kong local event-day hours."
    ),
    "",
    (
        "The exact join key is `event_date × market_id × decision_rule`. "
        "No missing value is imputed and no later market price or weather "
        "run is substituted."
    ),
    "",
    "## Sample flow",
    "",
    (
        f"- Candidate contract-rule rows: "
        f"{summary['candidate_contract_rule_rows']}"
    ),
    (
        f"- Market-price-ready rows: "
        f"{summary['market_price_ready_rows']}"
    ),
    (
        f"- Weather-path-ready rows: "
        f"{summary['weather_path_ready_rows']}"
    ),
    (
        f"- Exact common-support rows: "
        f"{summary['common_support_contract_rule_rows']}"
    ),
    (
        f"- Excluded rows: "
        f"{summary['excluded_contract_rule_rows']}"
    ),
    (
        f"- Complete common-support books: "
        f"{summary['common_support_complete_books']}"
    ),
    (
        f"- Market-price-missing rows: "
        f"{summary['market_price_missing_rows']}"
    ),
    (
        f"- Weather-path-not-ready rows: "
        f"{summary['weather_path_not_ready_rows']}"
    ),
    f"- Joint missing rows: {summary['joint_missing_rows']}",
    "",
    "## Support by decision rule",
    "",
    (
        "| Rule | Candidate rows | Market ready | Weather ready | "
        "Common rows | Common books | Excluded dates |"
    ),
    "|---|---:|---:|---:|---:|---:|---|",
]

for row in support_flow.sort_values(
    "decision_rule_order"
).itertuples(index=False):
    report_lines.append(
        "| {rule} | {candidate} | {market} | {weather} | "
        "{common} | {books} | {dates} |".format(
            rule=row.decision_rule,
            candidate=int(
                row.candidate_contract_rows
            ),
            market=int(
                row.market_price_ready_rows
            ),
            weather=int(
                row.weather_path_ready_rows
            ),
            common=int(
                row.common_support_contract_rows
            ),
            books=int(
                row.common_support_books
            ),
            dates=(
                row.excluded_dates
                if row.excluded_dates
                else ""
            ),
        )
    )

report_lines.extend(
    [
        "",
        "## Market binary scores on exact support",
        "",
        "| Rule | n | Mean Brier | Mean log score |",
        "|---|---:|---:|---:|",
    ]
)

all_binary = market_binary_summary.loc[
    market_binary_summary[
        "contract_event_type"
    ].eq("ALL")
].sort_values("decision_rule_order")

for row in all_binary.itertuples(index=False):
    report_lines.append(
        f"| {row.decision_rule} | {int(row.n)} | "
        f"{float(row.mean_brier):.8f} | "
        f"{float(row.mean_log_score):.8f} |"
    )

report_lines.extend(
    [
        "",
        "## Complete-book diagnostics",
        "",
        (
            "| Rule | Books | Mean normalised categorical log | "
            "Mean normalised multiclass Brier | "
            "Market modal contains winner | Modal tie rate | "
            "Deterministic bin hit |"
        ),
        "|---|---:|---:|---:|---:|---:|---:|",
    ]
)

for row in market_categorical_summary.sort_values(
    "decision_rule_order"
).itertuples(index=False):
    report_lines.append(
        "| {rule} | {books} | {log} | {brier} | "
        "{market_hit} | {tie_rate} | {weather_hit} |".format(
            rule=row.decision_rule,
            books=int(row.n_complete_books),
            log=fmt_float(
                row.mean_normalised_categorical_log_score
            ),
            brier=fmt_float(
                row.mean_normalised_multiclass_brier
            ),
            market_hit=fmt_float(
                row.market_modal_contains_actual_winner_rate
            ),
            tie_rate=fmt_float(
                row.market_modal_tie_rate
            ),
            weather_hit=fmt_float(
                row.deterministic_exact_contract_hit_rate
            ),
        )
    )

report_lines.extend(
    [
        "",
        "## Deterministic weather errors on exact support",
        "",
        (
            "| Rule | Dates | Mean error °C | MAE °C | RMSE °C | "
            "Underforecast rate |"
        ),
        "|---|---:|---:|---:|---:|---:|",
    ]
)

for row in weather_error_summary.sort_values(
    "decision_rule_order"
).itertuples(index=False):
    report_lines.append(
        "| {rule} | {dates} | {error} | {mae} | "
        "{rmse} | {under} |".format(
            rule=row.decision_rule,
            dates=int(
                row.n_common_support_dates
            ),
            error=fmt_float(row.mean_error_c),
            mae=fmt_float(row.mae_c),
            rmse=fmt_float(row.rmse_c),
            under=fmt_float(
                row.underforecast_rate
            ),
        )
    )

report_lines.extend(
    [
        "",
        "## Interpretation",
        "",
        (
            "The market probabilities are scored only on rows that also "
            "have an admissible complete deterministic forecast path. "
            "Market probabilities are normalised only for complete-book "
            "distributional diagnostics; raw prices remain preserved."
        ),
        "",
        (
            "Where multiple contracts share the highest market probability, "
            "the modal set is retained explicitly. Modal hit and overlap metrics "
            "are tie-aware; no market-ID tie-break is used."
        ),
        "",
        (
            "The deterministic forecast is evaluated as a point forecast "
            "through temperature error and selected-bin accuracy. It is not "
            "assigned a probabilistic score and no Gaussian bridge or "
            "artificial ensemble feature is introduced."
        ),
        "",
        (
            "This exact support is the canonical June input for subsequent "
            "local residual post-processing and expanded-sample freezing."
        ),
    ]
)

report_path = (
    REPORT_DIR
    / "18r_june_2026_market_weather_common_support_report.md"
)
report_path.write_text(
    "\n".join(report_lines) + "\n",
    encoding="utf-8",
)

manifest_rows: list[dict[str, Any]] = []

for root in (OUT_DIR, REPORT_DIR):
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        if path.name == "18r_june_2026_sha256_manifest.csv":
            continue

        manifest_rows.append(
            {
                "path": str(
                    path.relative_to(REPO_ROOT)
                ),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )

manifest_path = (
    OUT_DIR
    / "18r_june_2026_sha256_manifest.csv"
)
pd.DataFrame(manifest_rows).to_csv(
    manifest_path,
    index=False,
)

print(f"Report: {report_path.relative_to(REPO_ROOT)}")
print(f"Manifest entries: {len(manifest_rows)}")

Report: reports/18r_june_2026_market_weather_common_support/18r_june_2026_market_weather_common_support_report.md
Manifest entries: 13


In [11]:
print("Final support flow:")
display(support_flow)

print("Exact-support market scores:")
display(
    market_binary_summary.loc[
        market_binary_summary[
            "contract_event_type"
        ].eq("ALL")
    ]
)

print("Complete-book diagnostics:")
display(market_categorical_summary)

print("Exact-support weather errors:")
display(weather_error_summary)

print("Excluded rows by reason:")
display(
    support_exclusions_output.groupby(
        [
            "decision_rule",
            "support_exclusion_reason",
        ]
    )
    .size()
    .rename("rows")
    .reset_index()
    .sort_values(
        [
            "decision_rule",
            "support_exclusion_reason",
        ]
    )
)

print(f"Final verdict: {verdict}")

Final support flow:


,decision_rule,decision_rule_order,candidate_contract_rows,market_price_ready_rows,weather_path_ready_rows,common_support_contract_rows,market_price_missing_rows,weather_path_not_ready_rows,both_missing_rows,candidate_date_rule_books,common_support_books,common_support_dates,excluded_dates
0,24h_prior,0,330,297,330,297,33,0,0,30,27,2026-06-01|2026-06-02|2026-06-03|2026-06-04|20...,2026-06-06|2026-06-07|2026-06-08
1,12h_prior,1,330,308,330,308,22,0,0,30,28,2026-06-01|2026-06-02|2026-06-03|2026-06-04|20...,2026-06-06|2026-06-07
2,6h_prior,2,330,330,319,319,0,11,0,30,29,2026-06-01|2026-06-02|2026-06-03|2026-06-04|20...,2026-06-24
3,event_day_open,3,330,330,330,330,0,0,0,30,30,2026-06-01|2026-06-02|2026-06-03|2026-06-04|20...,


Exact-support market scores:


,decision_rule,decision_rule_order,contract_event_type,n,n_dates,mean_brier,median_brier,mean_log_score,median_log_score,mean_market_probability,outcome_rate,median_price_staleness_hours,p95_price_staleness_hours
0,24h_prior,0,ALL,297,27,0.058715,0.000380,0.184025,0.019693,0.094088,0.090909,0.998056,0.999167
4,12h_prior,1,ALL,308,28,0.058737,0.000298,0.184717,0.017401,0.093131,0.090909,0.998333,0.999167
8,6h_prior,2,ALL,319,29,0.056012,0.000182,0.177429,0.013592,0.093580,0.090909,0.998333,0.999167
12,event_day_open,3,ALL,330,30,0.055751,0.000086,0.177376,0.009293,0.094024,0.090909,0.998333,0.999167


Complete-book diagnostics:


...

Exact-support weather errors:


,decision_rule,decision_rule_order,n_common_support_dates,mean_forecast_daily_max_c,mean_hko_daily_max_c,mean_error_c,mae_c,rmse_c,median_absolute_error_c,underforecast_rate,deterministic_exact_contract_hit_rate
0,24h_prior,0,27,29.407407,31.181481,-1.774074,1.833333,2.016139,1.70,0.962963,0.000000
1,12h_prior,1,28,29.425000,31.171429,-1.746429,1.746429,1.889539,1.65,1.000000,0.071429
2,6h_prior,2,29,29.462069,31.117241,-1.655172,1.758621,1.978854,1.70,0.931034,0.103448
3,event_day_open,3,30,29.516667,31.173333,-1.656667,1.750000,2.011384,1.55,0.966667,0.066667


Excluded rows by reason:


,decision_rule,support_exclusion_reason,rows
0,12h_prior,MARKET_PRICE_MISSING,22
1,24h_prior,MARKET_PRICE_MISSING,33
2,6h_prior,WEATHER_PATH_NOT_READY,11


Final verdict: PASS
